In [1]:
from dotenv import load_dotenv

load_dotenv()
print("Notebook ready")

Notebook ready


In [2]:
%pip install anthropic

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

In [38]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


# Chat helper — temperature omitted (deprecated on claude-sonnet-5)
def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)

    # claude-sonnet-5 can return ThinkingBlock before TextBlock
    text_parts = []
    for block in message.content:
        if getattr(block, "type", None) == "text":
            text_parts.append(block.text)
    return "".join(text_parts)


In [39]:
import json

def extract_json(text):
    """Parse JSON from model text that may include fences or extra prose."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[-1]
        if cleaned.endswith("```"):
            cleaned = cleaned.rsplit("```", 1)[0]
        cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    for open_ch, close_ch in (("{", "}"), ("[", "]")):
        start = cleaned.find(open_ch)
        end = cleaned.rfind(close_ch)
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(cleaned[start : end + 1])
            except json.JSONDecodeError:
                continue

    raise json.JSONDecodeError("No valid JSON found in model response", cleaned, 0)


def generate_dataset():
    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    [
      {
        "task": "Description of task"
      }
    ]

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code
    * Not include extra text or comments the output should be a valid JSON array
    Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    return extract_json(text)


In [40]:
dataset = generate_dataset()
print(dataset)

[{'task': 'Write a regex pattern to validate and extract the account ID, region, and resource type from an AWS ARN string (e.g., arn:aws:s3:::my-bucket or arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0)'}, {'task': 'Write a Python function that takes an S3 bucket policy as input and checks whether it allows public read access by inspecting the Principal and Action fields'}, {'task': 'Create a JSON object representing an AWS IAM policy that grants a user read-only access to a specific S3 bucket and its objects'}]


In [41]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [32]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""


    print(test_case["task"])
    prompt = f"""
Please solve the following task:



{test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [35]:
def extract_json(text):
    """Parse JSON from model text that may include fences or extra prose."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[-1]
        if cleaned.endswith("```"):
            cleaned = cleaned.rsplit("```", 1)[0]
        cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Fall back to the first complete {...} or [...] object
    for open_ch, close_ch in (("{", "}"), ("[", "]")):
        start = cleaned.find(open_ch)
        end = cleaned.rfind(close_ch)
        if start != -1 and end != -1 and end > start:
            try:
                return json.loads(cleaned[start : end + 1])
            except json.JSONDecodeError:
                continue

    raise json.JSONDecodeError("No valid JSON found in model response", cleaned, 0)


def run_test_case(test_case):
    """Calls run_prompt with the test case and returns the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }


def grade_by_model(test_case, output):
    # Keep the solution short so the grader can finish its JSON
    solution = output if len(output) <= 2500 else output[:2500] + "\n...[truncated]..."

    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {solution}

Return ONLY a compact JSON object (no markdown, no extra text) with these keys:
- "strengths": array of 1-3 short strings
- "weaknesses": array of 1-3 short strings
- "reasoning": one short paragraph
- "score": number from 1 to 10

Keep every string under 20 words so the JSON stays complete.
"""

    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages)
    print(eval_text)
    return extract_json(eval_text)


In [34]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [30]:
def run_eval(dataset):
    """Load the dataset, run the test cases, and return the results"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results
    

In [42]:
with open('dataset.json', 'r') as f:
    dataset = json.load(f)

results = run_eval(dataset)

Write a regex pattern to validate and extract the account ID, region, and resource type from an AWS ARN string (e.g., arn:aws:s3:::my-bucket or arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0)
{"strengths":["Clear breakdown table and explanation of ARN structure","Named capture groups improve readability","Python implementation handles resource splitting more robustly than the regex alone"],"weaknesses":["Top-level regex fails on the exact s3 bucket example from the task since it requires a '/' or ':' separator after resource_type","Inconsistency between the headline regex and the working regex used in the Python code","Truncated/incomplete code output leaves test results unverified"],"reasoning":"The solution offers good structural explanation and thoughtful use of named groups, but the primary regex pattern presented as the answer critically fails to match the simplest example (arn:aws:s3:::my-bucket) given in the task, since it mandates a resource-type/resource-id se

In [43]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS ARN Regex Pattern\n\n## Regex Pattern\n\n```regex\n^arn:(?P<partition>aws[a-zA-Z-]*):(?P<service>[a-zA-Z0-9-]+):(?P<region>[a-zA-Z0-9-]*):(?P<account_id>\\d{12})?:(?P<resource_type>[a-zA-Z0-9-]+)[/:](?P<resource_id>[a-zA-Z0-9-_./]+)$\n```\n\n## Breakdown of ARN Structure\n\n```\narn:partition:service:region:account-id:resource-type/resource-id\narn:partition:service:region:account-id:resource-type:resource-id\n```\n\n## Detailed Pattern Explanation\n\n| Component | Pattern | Description |\n|-----------|---------|--------------|\n| `arn:` | Literal | Fixed prefix |\n| `(?P<partition>aws[a-zA-Z-]*)` | e.g., `aws`, `aws-cn`, `aws-us-gov` | AWS partition |\n| `(?P<service>[a-zA-Z0-9-]+)` | e.g., `s3`, `ec2`, `iam` | AWS service name |\n| `(?P<region>[a-zA-Z0-9-]*)` | e.g., `us-east-1` (can be empty) | AWS region |\n| `(?P<account_id>\\d{12})?` | 12 digits (optional) | AWS account ID |\n| `(?P<resource_type>[a-zA-Z0-9-]+)` | e.g., `instance`, `bucket` | Resource t